In [3]:
# Reference implementation: <https://github.com/isec-tugraz/ca-tip5family-monolith>.

from hashlib import sha256
from blake3 import blake3

SHA256 = lambda x: sha256(x).digest()
Blake3 = lambda x, n: blake3(x).digest(n)

LOOKUP_TABLE = [
    0, 7, 26, 63, 124, 215, 85, 254, 214, 228, 45, 185, 140, 173, 33, 240, 29, 177, 176, 32, 8,
    110, 87, 202, 204, 99, 150, 106, 230, 14, 235, 128, 213, 239, 212, 138, 23, 130, 208, 6, 44,
    71, 93, 116, 146, 189, 251, 81, 199, 97, 38, 28, 73, 179, 95, 84, 152, 48, 35, 119, 49, 88,
    242, 3, 148, 169, 72, 120, 62, 161, 166, 83, 175, 191, 137, 19, 100, 129, 112, 55, 221, 102,
    218, 61, 151, 237, 68, 164, 17, 147, 46, 234, 203, 216, 22, 141, 65, 57, 123, 12, 244, 54, 219,
    231, 96, 77, 180, 154, 5, 253, 133, 165, 98, 195, 205, 134, 245, 30, 9, 188, 59, 142, 186, 197,
    181, 144, 92, 31, 224, 163, 111, 74, 58, 69, 113, 196, 67, 246, 225, 10, 121, 50, 60, 157, 90,
    122, 2, 250, 101, 75, 178, 159, 24, 36, 201, 11, 243, 132, 198, 190, 114, 233, 39, 52, 21, 209,
    108, 238, 91, 187, 18, 104, 194, 37, 153, 34, 200, 143, 126, 155, 236, 118, 64, 80, 172, 89,
    94, 193, 135, 183, 86, 107, 252, 13, 167, 206, 136, 220, 207, 103, 171, 160, 76, 182, 227, 217,
    158, 56, 174, 4, 66, 109, 139, 162, 184, 211, 249, 47, 125, 232, 117, 43, 16, 42, 127, 20, 241,
    25, 149, 105, 156, 51, 53, 168, 145, 247, 223, 79, 78, 226, 15, 222, 82, 115, 70, 210, 27, 41,
    1, 170, 40, 131, 192, 229, 248, 255,
]

ROUND_CONSTANTS = [
    13630775303355457758, 16896927574093233874, 10379449653650130495, 1965408364413093495,
    15232538947090185111, 15892634398091747074, 3989134140024871768, 2851411912127730865,
    8709136439293758776,  3694858669662939734,  12692440244315327141, 10722316166358076749,
    12745429320441639448, 17932424223723990421, 7558102534867937463,  15551047435855531404,
    17532528648579384106, 5216785850422679555,  15418071332095031847, 11921929762955146258,
    9738718993677019874,  3464580399432997147,  13408434769117164050, 264428218649616431,
    4436247869008081381,  4063129435850804221,  2865073155741120117,  5749834437609765994,
    6804196764189408435,  17060469201292988508, 9475383556737206708,  12876344085611465020,
    13835756199368269249, 1648753455944344172,  9836124473569258483,  12867641597107932229,
    11254152636692960595, 16550832737139861108, 11861573970480733262, 1256660473588673495,
    13879506000676455136, 10564103842682358721, 16142842524796397521, 3287098591948630584,
    685911471061284805,   5285298776918878023,  18310953571768047354, 3142266350630002035,
    549990724933663297,   4901984846118077401,  11458643033696775769, 8706785264119212710,
    12521758138015724072, 11877914062416978196, 11333318251134523752, 3933899631278608623,
    16635128972021157924, 10291337173108950450, 4142107155024199350,  16973934533787743537,
    11068111539125175221, 17546769694830203606, 5315217744825068993,  4609594252909613081,
    3350107164315270407,  17715942834299349177, 9600609149219873996,  12894357635820003949,
    4597649658040514631,  7735563950920491847,  1663379455870887181,  13889298103638829706,
    7375530351220884434,  3502022433285269151,  9231805330431056952,  9252272755288523725,
    10014268662326746219, 15565031632950843234, 1209725273521819323,  6024642864597845108,
]

MDS_MATRIX_FIRST_COLUMN = [
    61402,  1108,  28750, 33823, 7454,  43244, 53865, 12034,
    56951, 27521, 41351, 40901, 12021, 59689, 26798, 17845,
]

MDS_MATRIX_FIRST_COLUMN_RESCUE = [
    7, 23, 8, 26, 13, 10, 9, 7, 6, 22, 21, 8,
]


def Gen_RoundConstant(p, string, m, N):
    """Generate round constants by hashing a domain-separated seed.

    Parameters
    ----------
    p      : modulus of the prime field GF(p).
    string : ASCII seed identifying the instance (e.g. 'Tip5', 'Tip4').
    m      : state size (number of constants per round).
    N      : number of rounds.

    Returns
    -------
    A list of length N, each entry a vector of length m over GF(p).
    Output has been cross-checked against the published Tip5 round constants.
    """
    RC = [[0 for _ in range(m)] for _ in range(N)]
    Fp = FiniteField(p)

    # Each constant is computed in Montgomery-style: first as an integer mod p,
    # then multiplied by R^-1 = (2^64)^-1 to undo the Montgomery factor.
    mul = Fp(2**64)
    mul_inverse = mul^-1

    for r in range(N):
        for i in range(m):
            # Domain separation: the (r, i) pair is encoded as a single byte.
            i_value = int(i + r * m)
            bytes_val = i_value.to_bytes(1, 'big')
            seed_str = "{}".format(string)
            # Concatenate seed and index bytes, then hash via Blake3.
            byte_str = Blake3(bytes(seed_str, "ascii") + bytes_val, m)

            # Interpret hash bytes as a little-endian integer mod p.
            integer = Fp(sum((2**8)**j * byte_str[j] for j in range(m)))
            integer *= mul_inverse
            RC[r][i] = integer
        RC[r] = vector(Fp, RC[r])

    return RC


In [1]:
# ---------------------------------------------------------------------------
# Sage implementation of the Tip5 sponge permutation (also used to build Tip4
# and Tip4' by instantiating with different parameters).
# ---------------------------------------------------------------------------

from sage.crypto.sbox import SBox
from scipy.linalg import circulant
from random import randint


class Tip5:
    """Tip5 / Tip4 / Tip4' permutation over GF(2^64 - 2^32 + 1)."""

    def __init__(self,
                 p=2**64 - 2**32 + 1,
                 state_size=16,
                 num_rounds=5,
                 alpha=7,
                 name="TIP",
                 num_split_and_lookup=4,
                 rate=10,
                 capacity=6,
                 digest_length=5,
                 initial_capacity_value=1,
                 R=2**64):
        assert ceil(log(p, 2)) == 64
        self.p = p                       # 0xffffffff00000001
        self.p_orig = 0xffffffff00000001
        self.field = GF(self.p)
        self.to_field   = lambda x: self.field(x)
        self.from_field = lambda x: Integer(x)
        # Random field element generator; bounded to make sure the S S-box is well defined.
        self.random_element = lambda: self.to_field(
            randint(0, min(self.p_orig, self.p) - 1)
        )

        self.name = name

        max_state_size, max_num_rounds = 16, 5
        assert state_size <= max_state_size and num_rounds <= max_num_rounds
        self.m = state_size
        self.N = num_rounds
        self.s = num_split_and_lookup

        # ------------------------------------------------------------------
        # Non-linear layer components -- S (split-and-lookup).
        # ------------------------------------------------------------------
        self.lut     = SBox([x for x in LOOKUP_TABLE])  # 8-bit LUT.
        self.lut_inv = self.lut.inverse()
        self.R       = self.to_field(R)
        self.R_inv   = self.R**(-1)

        # ------------------------------------------------------------------
        # Non-linear layer components -- T (power map x^alpha).
        # ------------------------------------------------------------------
        assert gcd(alpha, self.p - 1) == 1
        self.alpha     = alpha
        self.alpha_inv = inverse_mod(self.alpha, self.p - 1)

        # ------------------------------------------------------------------
        # Linear layer components -- circulant MDS matrix and round constants.
        # ------------------------------------------------------------------
        if self.m == max_state_size:
            self.mds_matrix = Matrix(
                self.field, nrows=self.m, ncols=self.m,
                entries=circulant(MDS_MATRIX_FIRST_COLUMN),
            )
        else:
            self.mds_matrix = Matrix(
                self.field, nrows=self.m, ncols=self.m,
                entries=circulant(MDS_MATRIX_FIRST_COLUMN_RESCUE),
            )
        self.mds_matrix_inv = self.mds_matrix.inverse()

        # Diagonal R-matrix: scales the first s positions by R (Montgomery),
        # leaves the remaining (m - s) T-box positions unchanged.
        self.r_matrix     = diagonal_matrix(
            self.field, self.m,
            vector([self.R] * self.s + [1] * (self.m - self.s)),
        )
        self.r_matrix_inv = diagonal_matrix(
            self.field, self.m,
            vector([self.R_inv] * self.s + [1] * (self.m - self.s)),
        )
        
        self.rcons = Gen_RoundConstant(self.p, self.name, self.m, self.N)

        # ------------------------------------------------------------------
        # Hash / sponge configuration.
        # ------------------------------------------------------------------
        assert self.m == rate + capacity
        self.r = rate
        self.c = capacity
        assert self.m >= digest_length
        self.d = digest_length
        self.c_value = self.to_field(initial_capacity_value)

    # ----------------------------------------------------------------------
    # Pretty-printing.
    # ----------------------------------------------------------------------
    def __str__(self):
        return (
            f"{self.name} over {self.field}\n"
            f"Rate = {self.r}, Capacity = {self.c}, Digest length = {self.d}, "
            f"Number of split and look S-Boxes = {self.s}, Alpha = {self.alpha}\n"
            f"Number of rounds = {self.N}, Initial capacity value = {self.c_value}, "
            f"R for Montgomery = {self.R}\n"
        )

    # ----------------------------------------------------------------------
    # Hash entry-point (sponge absorb + squeeze of a single rate block).
    # ----------------------------------------------------------------------
    def __call__(self, message):
        # Only the rate part is chosen by the caller; the capacity is fixed.
        assert len(message) == self.r
        input_state = vector(self.field, message + [self.c_value] * self.c)
        result = self.eval_with_intermediate_states(input_state)
        hash_value = list(result[-1])[:self.d]
        return hash_value

    def eval_with_intermediate_states(self, input_state, N=None):
        """Run the permutation forward, returning the state after every round."""
        if len(input_state) == self.r:
            input_state = input_state + [self.c_value] * self.c
        input_state = vector(self.field, input_state)
        assert len(input_state) == self.m
        if N is None:
            N = self.N

        result = [input_state]
        for r in range(N):
            state = result[-1]
            state = self.nonlinear_layer(state)
            state = self.linear_layer(state, r)
            result.append(state)
        return result

    def eval_inv_with_intermediate_states(self, output_state):
        """Run the permutation backward; both rate and capacity must be known."""
        assert len(output_state) == self.m
        result = [vector(self.field, output_state)]
        for r in range(self.N - 1, -1, -1):
            state = result[-1]
            state = self.linear_layer(state, r, inv=True)
            state = self.nonlinear_layer(state, inv=True)
            result.append(state)
        return result

    # ----------------------------------------------------------------------
    # Non-linear layer: S (split + 8-bit LUT) and T (power map).
    # ----------------------------------------------------------------------
    def decompose(self, x):
        """Split a 64-bit field element into 8 bytes, most-significant first."""
        x = self.from_field(x)
        parts = []
        for si in [8] * 8:                # 64 bits = 8 x 8 bits
            parts.append(x & (2**si - 1))
            x = x >> si
        return parts[::-1]

    def compose(self, parts):
        """Inverse of decompose: reassemble 8 bytes into a field element."""
        x = parts[0]
        for xi, si in zip(parts[1:], [8] * 7):
            x = x << si
            x += xi
        assert x < self.p
        return self.to_field(x)

    def S(self, x, inv=False):
        """Split-and-lookup S-box, including the Montgomery conversion."""
        x = self.R * x                                    # canonical -> Montgomery
        parts = self.decompose(x)
        sub_parts = [self.lut_inv(xi) if inv else self.lut(xi) for xi in parts]
        y = self.compose(sub_parts)
        y = self.R_inv * y                                # Montgomery -> canonical
        return y

    def S_without_R(self, x, inv=False):
        """S-box variant that skips the Montgomery conversion (analysis helper)."""
        parts = self.decompose(x)
        sub_parts = [self.lut_inv(xi) if inv else self.lut(xi) for xi in parts]
        return self.compose(sub_parts)

    def T(self, x, inv=False):
        return x**self.alpha_inv if inv else x**self.alpha

    def nonlinear_layer(self, state, inv=False):
        sub_state = copy(state)
        for i in range(self.m):
            if i < self.s:
                sub_state[i] = self.S(sub_state[i], inv)
            else:
                sub_state[i] = self.T(sub_state[i], inv)
        return sub_state

    # ----------------------------------------------------------------------
    # Linear layer: MDS multiplication and round-constant addition.
    # ----------------------------------------------------------------------
    def multiply_mds_matrix(self, state, inv=False):
        return self.mds_matrix_inv * state if inv else self.mds_matrix * state

    def add_rcons(self, state, r, inv=False):
        return state - self.rcons[r] if inv else state + self.rcons[r]

    def linear_layer(self, state, r, inv=False):
        if inv:
            state = self.add_rcons(state, r, inv)
            state = self.multiply_mds_matrix(state, inv)
        else:
            state = self.multiply_mds_matrix(state, inv)
            state = self.add_rcons(state, r, inv)
        return state


In [6]:
# ---------------------------------------------------------------------------
# Build the three concrete instances we will analyse.
# ---------------------------------------------------------------------------

tip5  = Tip5(name="Tip5")
tip4  = Tip5(name="Tip4",  rate=12, capacity=4, digest_length=4)
tip4p = Tip5(name="Tip4'", state_size=12, rate=8, capacity=4, digest_length=4)

print(tip5)
print(tip4)
print(tip4p)


Tip5 over Finite Field of size 18446744069414584321
Rate = 10, Capacity = 6, Digest length = 5, Number of split and look S-Boxes = 4, Alpha = 7
Number of rounds = 5, Initial capacity value = 1, R for Montgomery = 4294967295

Tip4 over Finite Field of size 18446744069414584321
Rate = 12, Capacity = 4, Digest length = 4, Number of split and look S-Boxes = 4, Alpha = 7
Number of rounds = 5, Initial capacity value = 1, R for Montgomery = 4294967295

Tip4' over Finite Field of size 18446744069414584321
Rate = 8, Capacity = 4, Digest length = 4, Number of split and look S-Boxes = 4, Alpha = 7
Number of rounds = 5, Initial capacity value = 1, R for Montgomery = 4294967295



# Searching for the minimum number of active S-boxes



In [17]:
# ---------------------------------------------------------------------------
# computeCoeffMatrix_Tip5
#
# For Tip5 (capacity = 6, fixed_rows = range(10, 16)), enumerate every
# combination of:
#   - exactly one column from the S-box block (cols 0..3); the other three
#     S-box columns are placed first in the selection so they are eliminated
#     during echelon reduction;
#   - three columns from the T-box block (cols 4..15).
#
# For each selection the routine returns the last column of the row-reduced
# 6x7 submatrix of M^{-1}, which gives the coefficients that map a single
# 8-bit difference at the chosen S-box column to the three T-box positions.
# ---------------------------------------------------------------------------

from scipy.linalg import circulant
from itertools import product, combinations


def computeCoeffMatrix_Tip5(cipher):
    """Coefficient matrix enumeration for the original Tip5 (capacity = 6)."""
    m     = cipher.m
    RM    = cipher.r_matrix
    RInvM = cipher.r_matrix_inv
    mds   = cipher.mds_matrix
    M     = RM * mds * RInvM
    MInv  = M.inverse()

    all_subMInv       = []
    all_subMInv_coeff = []
    subMInv_dict      = {}

    # Rows of M^-1 corresponding to the capacity part of the state.
    fixed_rows = list(range(10, 16))

    # Columns of M^-1 corresponding to the S-box block and the T-box block.
    fixed_cols_range    = list(range(0, 4))
    variable_cols_range = list(range(4, 16))

    for combo in combinations(fixed_cols_range, 1):
        remaining  = sorted(set(fixed_cols_range) - set(combo))
        fixed_cols = list(combo) + remaining

        # All ways to pick three T-box columns alongside the four S-box columns.
        for extra_cols in combinations(variable_cols_range, 3):
            selected_cols = fixed_cols + list(extra_cols)

            # 6 x 7 submatrix of M^-1 with columns in reverse order so that
            # echelon reduction isolates the active S-box column as a pivot.
            subMInv = matrix(
                cipher.field, len(fixed_rows), 7,
                lambda i, j: MInv[fixed_rows[i]][selected_cols[6 - j]],
            )
            all_subMInv.append(subMInv)

            subMInv_ef = subMInv.echelon_form()

            # Coefficients reported as (p - last column) so that adding the
            # active-byte difference in GF(p) cancels the pivot column.
            subMInv_coeff = matrix(
                cipher.field, len(fixed_rows), 1,
                lambda i, j: cipher.p - subMInv_ef[i][6],
            )
            all_subMInv_coeff.append(subMInv_coeff)

            subMInv_dict[tuple(selected_cols)] = subMInv_coeff

    # The first 4 entries of each key are the S-box block (1 active + 3 in the
    # pivot positions); the last 3 rows of the coefficient matrix are the part
    # that actually feeds into the active T-box positions.
    for key, value in subMInv_dict.items():
        print("{} : {}".format(key[0:4], list(value[3:])))

    return subMInv_dict


computeCoeffMatrix_Tip5(cipher)


(0, 1, 2, 3) : [(11279794763508077807), (17536089927571604939), (13741015968529866384)]
(0, 1, 2, 3) : [(11614308219342252650), (18006113497905605890), (13561665458283758349)]
(0, 1, 2, 3) : [(14951082729797008661), (17621996816318266971), (2197220445918080219)]
(0, 1, 2, 3) : [(3817521751049186899), (7840172112785340048), (3268439939134554643)]
(0, 1, 2, 3) : [(13939321662293854630), (10821092064456734393), (12262236902938840406)]
(0, 1, 2, 3) : [(12286802146319583268), (10851943165073485760), (13665810940804281583)]
(0, 1, 2, 3) : [(4480380500192774915), (8672972171766393540), (14683271048643121180)]
(0, 1, 2, 3) : [(8293354632736126339), (15783906034018191861), (9227149572248303492)]
(0, 1, 2, 3) : [(11799733298166264147), (4431089093065991791), (3364023109032795452)]
(0, 1, 2, 3) : [(7631198354371747470), (4153733170803292195), (2453535919451202082)]
(0, 1, 2, 3) : [(4062486557154198102), (7810080191707516939), (4603790802306002257)]
(0, 1, 2, 3) : [(13405293157557655682), (1668897

{(0,
  1,
  2,
  3,
  4,
  5,
  6): [ 4956350808526882103]
 [ 4665103756170311014]
 [ 2879289109038866704]
 [11279794763508077807]
 [17536089927571604939]
 [13741015968529866384],
 (0,
  1,
  2,
  3,
  4,
  5,
  7): [10237959967116841113]
 [18064263092105378771]
 [10851096527413789968]
 [11614308219342252650]
 [18006113497905605890]
 [13561665458283758349],
 (0,
  1,
  2,
  3,
  4,
  5,
  8): [15345923676370113950]
 [15771416913268439205]
 [ 7328597717234196912]
 [14951082729797008661]
 [17621996816318266971]
 [ 2197220445918080219],
 (0,
  1,
  2,
  3,
  4,
  5,
  9): [10660577398576258518]
 [ 8906982547097511790]
 [ 5244600752599358022]
 [ 3817521751049186899]
 [ 7840172112785340048]
 [ 3268439939134554643],
 (0,
  1,
  2,
  3,
  4,
  5,
  10): [17287815777728411933]
 [ 2677837582611081272]
 [17678545296117733328]
 [13939321662293854630]
 [10821092064456734393]
 [12262236902938840406],
 (0,
  1,
  2,
  3,
  4,
  5,
  11): [11807429921459807806]
 [11935725017270685866]
 [1300145034569

In [18]:
# ---------------------------------------------------------------------------
# iterInputDiff: enumerate all 64-bit field elements whose byte representation
# has exactly `active_nb` non-zero bytes. Used as the set of single-limb input
# differences for the search.
# ---------------------------------------------------------------------------

def iterInputDiff(cipher, active_nb):
    """Return all distinct non-zero field elements with `active_nb` active bytes."""
    pos_combinations = list(combinations(range(8), active_nb))

    results = []
    for pos_set in pos_combinations:
        for free_vals in product(range(2**8), repeat=active_nb):
            bytes_list = [0] * 8
            for idx, pos in enumerate(pos_set):
                bytes_list[pos] = free_vals[idx]
            x_val = sum([b << (8 * i) for i, b in enumerate(bytes_list)])
            # Filter values that do not lie inside GF(p).
            if x_val < cipher.p:
                results.append(cipher.field(x_val))
                print(bytes_list, x_val)

    results = list(set(results))
    results.remove(cipher.field(0))
    print(len(results))

    return results


iterInputDiff(cipher, active_nb=1)


[0, 0, 0, 0, 0, 0, 0, 0] 0
[1, 0, 0, 0, 0, 0, 0, 0] 1
[2, 0, 0, 0, 0, 0, 0, 0] 2
[3, 0, 0, 0, 0, 0, 0, 0] 3
[4, 0, 0, 0, 0, 0, 0, 0] 4
[5, 0, 0, 0, 0, 0, 0, 0] 5
[6, 0, 0, 0, 0, 0, 0, 0] 6
[7, 0, 0, 0, 0, 0, 0, 0] 7
[8, 0, 0, 0, 0, 0, 0, 0] 8
[9, 0, 0, 0, 0, 0, 0, 0] 9
[10, 0, 0, 0, 0, 0, 0, 0] 10
[11, 0, 0, 0, 0, 0, 0, 0] 11
[12, 0, 0, 0, 0, 0, 0, 0] 12
[13, 0, 0, 0, 0, 0, 0, 0] 13
[14, 0, 0, 0, 0, 0, 0, 0] 14
[15, 0, 0, 0, 0, 0, 0, 0] 15
[16, 0, 0, 0, 0, 0, 0, 0] 16
[17, 0, 0, 0, 0, 0, 0, 0] 17
[18, 0, 0, 0, 0, 0, 0, 0] 18
[19, 0, 0, 0, 0, 0, 0, 0] 19
[20, 0, 0, 0, 0, 0, 0, 0] 20
[21, 0, 0, 0, 0, 0, 0, 0] 21
[22, 0, 0, 0, 0, 0, 0, 0] 22
[23, 0, 0, 0, 0, 0, 0, 0] 23
[24, 0, 0, 0, 0, 0, 0, 0] 24
[25, 0, 0, 0, 0, 0, 0, 0] 25
[26, 0, 0, 0, 0, 0, 0, 0] 26
[27, 0, 0, 0, 0, 0, 0, 0] 27
[28, 0, 0, 0, 0, 0, 0, 0] 28
[29, 0, 0, 0, 0, 0, 0, 0] 29
[30, 0, 0, 0, 0, 0, 0, 0] 30
[31, 0, 0, 0, 0, 0, 0, 0] 31
[32, 0, 0, 0, 0, 0, 0, 0] 32
[33, 0, 0, 0, 0, 0, 0, 0] 33
[34, 0, 0, 0, 0, 0, 0, 0] 34
[35, 

[1,
 2,
 3,
 4,
 5,
 6,
 7,
 8,
 9,
 10,
 11,
 12,
 13,
 14,
 15,
 16,
 17,
 18,
 19,
 20,
 21,
 22,
 23,
 24,
 25,
 26,
 27,
 28,
 29,
 30,
 31,
 32,
 33,
 34,
 35,
 36,
 37,
 38,
 39,
 40,
 41,
 42,
 43,
 44,
 45,
 46,
 47,
 48,
 49,
 50,
 51,
 52,
 53,
 54,
 55,
 56,
 57,
 58,
 59,
 60,
 61,
 62,
 63,
 64,
 65,
 66,
 67,
 68,
 69,
 70,
 71,
 72,
 73,
 74,
 75,
 76,
 77,
 78,
 79,
 80,
 81,
 82,
 83,
 84,
 85,
 86,
 87,
 88,
 89,
 90,
 91,
 92,
 93,
 94,
 95,
 96,
 97,
 98,
 99,
 100,
 101,
 102,
 103,
 104,
 105,
 106,
 107,
 108,
 109,
 110,
 111,
 112,
 113,
 114,
 115,
 116,
 117,
 118,
 119,
 120,
 121,
 122,
 123,
 124,
 125,
 126,
 127,
 128,
 129,
 130,
 131,
 132,
 133,
 134,
 135,
 136,
 137,
 138,
 139,
 140,
 141,
 142,
 143,
 144,
 145,
 146,
 147,
 148,
 149,
 150,
 151,
 152,
 153,
 154,
 155,
 156,
 157,
 158,
 159,
 160,
 161,
 162,
 163,
 164,
 165,
 166,
 167,
 168,
 169,
 170,
 171,
 172,
 173,
 174,
 175,
 176,
 177,
 178,
 179,
 180,
 181,
 182,
 183,
 184,
 185

In [19]:
# ---------------------------------------------------------------------------
# countActiveBytes: count the number of non-zero 8-bit limbs in a field
# element interpreted as an unsigned 64-bit integer.
# ---------------------------------------------------------------------------

def countActiveBytes(dx):
    """Return the number of non-zero bytes in the 64-bit representation of `dx`."""
    d = Integer(dx)
    count = 0
    for i in range(8):
        byte = (d >> (8 * i)) & 0xFF
        if byte != 0:
            count += 1
    return count


In [20]:
# ---------------------------------------------------------------------------
# countAllTrans: for every coefficient matrix produced above and every input
# difference in `results`, count the total number of active 8-bit S-boxes
# after one inverse linear layer (active_nb input limbs + the limbs that get
# activated in the last few rows of M^-1 * dx).
# ---------------------------------------------------------------------------

def countAllTrans(cipher, subMInv_dict, results, active_nb):
    """Print the minimum number of active S-boxes for each column selection."""
    min_bytes_all     = 0
    min_dx            = 0
    min_active_bytes  = []

    for key, value in subMInv_dict.items():
        pos          = key
        subMInv_coeff = value

        all_active_bytes = []
        for x in results:
            dx0 = matrix(cipher.field, 1, 1, x)
            dx  = subMInv_coeff * dx0
            # Only the last 3 rows correspond to active T-box positions in the
            # capacity = 4 / 7-column setup; sum their active-byte counts.
            active_bytes = 0
            for l in dx[-3:]:
                active_bytes += countActiveBytes(l[0])
            all_active_bytes.append(active_bytes + active_nb)

        min_active_bytes.append((key, min(all_active_bytes)))
        if min_bytes_all >= min(all_active_bytes) or min_bytes_all == 0:
            min_bytes_all = min(all_active_bytes)

    for entry in min_active_bytes:
        print(entry)

    print(f"min #sbox = {min_bytes_all}")


In [21]:
cipher       = tip5
active_nb    = 1
subMInv_dict = computeCoeffMatrix_Tip5_c5(cipher)
results      = iterInputDiff(cipher, active_nb)
countAllTrans(cipher, subMInv_dict, results, active_nb)


(0, 1, 2, 3, 4, 5, 6) : [(10903133897552206425), (647213097992005800), (15716595647509069072), (17804064203754819974)]
(0, 1, 2, 3, 4, 5, 7) : [(5087218298864685379), (8223325921925256697), (16308310909224635070), (8176058775818661059)]
(0, 1, 2, 3, 4, 5, 8) : [(17112183759636927245), (9573512566609432253), (9803834045036270155), (17935109945188149285)]
(0, 1, 2, 3, 4, 5, 9) : [(8284993228954628420), (2132088780798812626), (17851824606066754931), (4446985381633471085)]
(0, 1, 2, 3, 4, 5, 10) : [(11231658795720439250), (10238105892814644970), (9935705076450226778), (1703926700141972499)]
(0, 1, 2, 3, 4, 5, 11) : [(9724965179735284169), (5364906110936533924), (15175835816591470534), (13943034114437425311)]
(0, 1, 2, 3, 4, 5, 12) : [(12265333755767416816), (7473080399672570459), (5424396697845087558), (2656775369277348296)]
(0, 1, 2, 3, 4, 5, 13) : [(9944824630220312829), (11487893352058576933), (4301597852939429708), (8857837656754136635)]
(0, 1, 2, 3, 4, 5, 14) : [(6582917683745341310),